# 04 Gender AI Proficiency Gap Monitor



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()
RAW_SYNTHETIC_DIR = REPO_ROOT / "data" / "raw" / "synthetic"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

input_path = RAW_SYNTHETIC_DIR / "synthetic_ai_proficiency_survey_demo.csv"
print("Input:", input_path)
print("Available:", input_path.exists())

In [ ]:
if input_path.exists():
    df = pd.read_csv(input_path)
else:
    raise FileNotFoundError("Place synthetic_ai_proficiency_survey_demo.csv in data/raw/synthetic/")

print(df.shape)
df.head()

In [ ]:
def yes_no_to_bool(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map({
            "yes": True,
            "no": False,
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        })
        .fillna(False)
        .astype(bool)
    )

data = df.copy()

for col in ["ai_training_completed", "career_ai_tool_used", "academic_misconduct_flag"]:
    data[col] = yes_no_to_bool(data[col])

for col in [
    "weekly_genai_use_hours",
    "ai_confidence_score_1_5",
    "assessment_score_after_ai_integration",
]:
    data[col] = pd.to_numeric(data[col], errors="coerce")

## Overall Gender Summary

In [ ]:
gender_summary = (
    data.groupby("gender", dropna=False)
    .agg(
        student_count=("student_id", "count"),
        avg_weekly_genai_use_hours=("weekly_genai_use_hours", "mean"),
        avg_ai_confidence_score=("ai_confidence_score_1_5", "mean"),
        ai_training_completion_rate=("ai_training_completed", "mean"),
        career_ai_tool_usage_rate=("career_ai_tool_used", "mean"),
        avg_assessment_score_after_ai_integration=("assessment_score_after_ai_integration", "mean"),
        academic_misconduct_flag_rate=("academic_misconduct_flag", "mean"),
    )
    .reset_index()
)

for col in [
    "ai_training_completion_rate",
    "career_ai_tool_usage_rate",
    "academic_misconduct_flag_rate",
]:
    gender_summary[col] = (gender_summary[col] * 100).round(2)

for col in [
    "avg_weekly_genai_use_hours",
    "avg_ai_confidence_score",
    "avg_assessment_score_after_ai_integration",
]:
    gender_summary[col] = gender_summary[col].round(2)

gender_summary

## Gender × Subject Dashboard Dataset

In [ ]:
gap_monitor = (
    data.groupby(["gender", "subject_area"], dropna=False)
    .agg(
        student_count=("student_id", "count"),
        avg_weekly_genai_use_hours=("weekly_genai_use_hours", "mean"),
        avg_ai_confidence_score=("ai_confidence_score_1_5", "mean"),
        ai_training_completion_rate=("ai_training_completed", "mean"),
        career_ai_tool_usage_rate=("career_ai_tool_used", "mean"),
        avg_assessment_score_after_ai_integration=("assessment_score_after_ai_integration", "mean"),
        academic_misconduct_flag_rate=("academic_misconduct_flag", "mean"),
    )
    .reset_index()
)

for col in [
    "ai_training_completion_rate",
    "career_ai_tool_usage_rate",
    "academic_misconduct_flag_rate",
]:
    gap_monitor[col] = (gap_monitor[col] * 100).round(2)

for col in [
    "avg_weekly_genai_use_hours",
    "avg_ai_confidence_score",
    "avg_assessment_score_after_ai_integration",
]:
    gap_monitor[col] = gap_monitor[col].round(2)

gap_monitor["monitoring_note"] = "Synthetic subgroup dashboard data. Not real student data."

gap_monitor.to_csv(PROCESSED_DIR / "gender_ai_gap_monitoring.csv", index=False)
gap_monitor.head(10)

In [ ]:
plot_df = gender_summary.sort_values("avg_weekly_genai_use_hours", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df["gender"], plot_df["avg_weekly_genai_use_hours"])
plt.title("Synthetic AI Proficiency Monitor: Weekly GenAI Use by Gender")
plt.xlabel("Gender")
plt.ylabel("Average weekly GenAI use hours")
plt.tight_layout()
plt.show()

In [ ]:
plot_df = gender_summary.sort_values("ai_training_completion_rate", ascending=False)

plt.figure(figsize=(8, 5))
plt.bar(plot_df["gender"], plot_df["ai_training_completion_rate"])
plt.title("Synthetic AI Proficiency Monitor: AI Training Completion by Gender")
plt.xlabel("Gender")
plt.ylabel("Training completion rate (%)")
plt.tight_layout()
plt.show()